## 05 — TF-IDF Optimization Experiments (Response + Numeric)

**Goal:** Improve over the strong baseline from Notebook 03 (TF-IDF(response) + numeric → Logistic Regression), while keeping:
- identical data loading (`load_splits`)
- identical evaluation protocol (`evaluate_split`, `metrics_table`)
- clean, reproducible experiment tracking

We proceed in 3 phases:
1) Reproduce the baseline (sanity check)
2) Tune TF-IDF + Logistic Regression in a controlled way (CV on train only)
3) Evaluate best configs on val/test and compare gains


#### Cell 1 — Bootstrap (same pattern as other notebooks)

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.experiment import seed_everything, make_run_dir

# Reproducibility
SEED = 42
seed_everything(SEED)

# Output directory
REPORTS_DIR = ROOT / "reports"
RUN_DIR = make_run_dir(REPORTS_DIR, "nb05_tfidf_optimization", timestamp=False)

print(f"Output directory: {RUN_DIR.relative_to(ROOT)}")

#### Cell 2 — Imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV

from src.data.load_splits import load_splits
from src.features.response_features import add_numeric_feature_columns, get_numeric_feature_cols
from src.utils.eval import evaluate_split, metrics_table

#### Cell 3 — Load splits (locked convention)



In [26]:
train_df, val_df, test_df = load_splits(root=ROOT)

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head(2)

(51647, 7) (6425, 7) (6435, 7)


,id,group_id,task,prompt,response,label,context
0,dialogue_1_gt,dialogue_1,dialogue,[Human]: Could you recommend any books like Th...,yes and he also produced White Oleander and it...,0,The Secret Life of Bees has genre Teen drama. ...
1,dialogue_1_hall,dialogue_1,dialogue,[Human]: Could you recommend any books like Th...,"No, I don't think Hunt Lowry was involved in A...",1,The Secret Life of Bees has genre Teen drama. ...


#### Cell 4 — Add numeric features

In [27]:
train_df_num = add_numeric_feature_columns(train_df.copy())
val_df_num   = add_numeric_feature_columns(val_df.copy())
test_df_num  = add_numeric_feature_columns(test_df.copy())

num_cols = get_numeric_feature_cols(train_df_num)   # <-- fix from the error you hit earlier
print("Number of numeric features:", len(num_cols))
print("First 15 numeric cols:", num_cols[:15])

Number of numeric features: 10
First 15 numeric cols: ['resp_n_chars', 'resp_n_words', 'resp_n_punct', 'resp_has_multi_excl', 'resp_has_multi_q', 'resp_has_ellipsis', 'resp_n_numbers', 'resp_n_uncertainty', 'resp_punct_per_word', 'resp_numbers_per_word']


#### Cell 5 — Experiment helpers

In [ ]:
def make_tfidf_lr_pipeline(
    tfidf_params: dict | None = None,
    lr_params: dict | None = None,
    *,
    response_col: str = "response",
    numeric_cols: list[str] | None = None,
    random_state: int = SEED,
):
    """
    TF-IDF on response + scaled numeric → Logistic Regression
    All params are applied INSIDE the pipeline (no leakage).
    """
    tfidf_params = tfidf_params or {}
    lr_params = lr_params or {}

    if numeric_cols is None:
        raise ValueError("numeric_cols must be provided")

    tfidf = TfidfVectorizer(**tfidf_params)

    # Note: with_mean=False is required for sparse matrices
    num_scaler = StandardScaler(with_mean=False)

    preprocess = ColumnTransformer(
        transformers=[
            ("tfidf_resp", tfidf, response_col),
            ("num", num_scaler, numeric_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

    clf = LogisticRegression(
        max_iter=2000,
        random_state=random_state,
        **lr_params,
    )

    return Pipeline([
        ("preprocess", preprocess),
        ("clf", clf),
    ])


def eval_all_splits(name: str, model, train_df, val_df, test_df):
    X_train, y_train = train_df, train_df["label"].values
    X_val,   y_val   = val_df,   val_df["label"].values
    X_test,  y_test  = test_df,  test_df["label"].values

    rows = []
    rows.append(evaluate_split(f"train_{name}", model, X_train, y_train))
    rows.append(evaluate_split(f"val_{name}",   model, X_val,   y_val))
    rows.append(evaluate_split(f"test_{name}",  model, X_test,  y_test))
    return metrics_table(rows)

#### Cell 6 — Phase 1: Reproduce baseline

In [29]:
baseline_tfidf = dict(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

baseline_lr = dict(
    C=1.0,
    solver="lbfgs",
    class_weight=None,
)

baseline_model = make_tfidf_lr_pipeline(
    tfidf_params=baseline_tfidf,
    lr_params=baseline_lr,
    numeric_cols=num_cols,
)

baseline_model.fit(train_df_num, train_df_num["label"].values)
baseline_metrics = eval_all_splits("tfidf+num_baseline", baseline_model, train_df_num, val_df_num, test_df_num)
baseline_metrics


train_tfidf+num_baseline metrics:
  accuracy = 0.8919
  f1       = 0.8821
  precision= 0.8958
  recall   = 0.8688
  confusion matrix:
[[25176  2429]
 [ 3154 20888]]

val_tfidf+num_baseline metrics:
  accuracy = 0.8327
  f1       = 0.8191
  precision= 0.8198
  recall   = 0.8184
  confusion matrix:
[[2916  535]
 [ 540 2434]]

test_tfidf+num_baseline metrics:
  accuracy = 0.8275
  f1       = 0.8151
  precision= 0.8103
  recall   = 0.8200
  confusion matrix:
[[2878  573]
 [ 537 2447]]


,split,accuracy,f1,precision,recall
0,train_tfidf+num_baseline,0.891901,0.882113,0.895827,0.868813
1,val_tfidf+num_baseline,0.832685,0.819115,0.819805,0.818426
2,test_tfidf+num_baseline,0.827506,0.815123,0.810265,0.820040


#### Cell 7 — Phase 2: Controlled tuning (GridSearchCV on TRAIN only)
We tune ONLY TF-IDF + LR hyperparams, using CV on train.

Validation split is kept untouched for model selection sanity check.

In [ ]:
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)

search_model = make_tfidf_lr_pipeline(
    tfidf_params=baseline_tfidf,
    lr_params=baseline_lr,
    numeric_cols=num_cols,
)

# --------------------
# 7A) Small WORD grid
# --------------------
word_grid_small = {
    "preprocess__tfidf_resp__analyzer": ["word"],
    "preprocess__tfidf_resp__ngram_range": [(1, 2)],   # keep fixed first
    "preprocess__tfidf_resp__min_df": [2, 5],
    "preprocess__tfidf_resp__max_df": [0.95],
    "preprocess__tfidf_resp__sublinear_tf": [True],
    "preprocess__tfidf_resp__norm": ["l2"],
    "clf__C": [0.5, 1.0, 2.0, 4.0],
    "clf__class_weight": [None, "balanced"],
}

grid_word = GridSearchCV(
    estimator=search_model,
    param_grid=word_grid_small,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid_word.fit(
    train_df_num,
    train_df_num["label"].values,
    groups=train_df_num["group_id"].values,
)

print("\n[WORD] Best CV F1:", grid_word.best_score_)
print("[WORD] Best params:", grid_word.best_params_)


# ---------------------
# 7B) Small CHAR grid
# ---------------------
char_grid_small = {
    "preprocess__tfidf_resp__analyzer": ["char_wb"],
    "preprocess__tfidf_resp__ngram_range": [(3, 5), (4, 6)],
    "preprocess__tfidf_resp__min_df": [2, 5],
    "preprocess__tfidf_resp__max_df": [0.95],
    "preprocess__tfidf_resp__sublinear_tf": [True],
    "preprocess__tfidf_resp__norm": ["l2"],
    "clf__C": [0.5, 1.0, 2.0, 4.0],
    "clf__class_weight": [None, "balanced"],
}

grid_char = GridSearchCV(
    estimator=search_model,
    param_grid=char_grid_small,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid_char.fit(
    train_df_num,
    train_df_num["label"].values,
    groups=train_df_num["group_id"].values,
)

print("\n[CHAR] Best CV F1:", grid_char.best_score_)
print("[CHAR] Best params:", grid_char.best_params_)


# ------------------------
# 7C) Choose the best one
# ------------------------
if grid_word.best_score_ >= grid_char.best_score_:
    best_search = grid_word
    best_tag = "best_from_word_grid"
else:
    best_search = grid_char
    best_tag = "best_from_char_grid"

best_model = best_search.best_estimator_
print(f"\nSelected: {best_tag} | CV F1 = {best_search.best_score_:.4f}")

## Conclusions — TF-IDF Optimization

- Reproduced the strong baseline from Notebook 03 (TF-IDF(response)+numeric).
- Conducted controlled hyperparameter tuning and character n-gram experiments.
- No configuration improved validation or test performance beyond the baseline.
- This suggests the classical feature-based model is close to its performance ceiling on the dataset.
- Motivates the use of task-adapted neural models in the next stage.


In [ ]:
import pandas as pd

out = pd.DataFrame([
    {
        "grid": "word",
        "best_cv_f1": grid_word.best_score_,
        **grid_word.best_params_,
    },
    {
        "grid": "char_wb",
        "best_cv_f1": grid_char.best_score_,
        **grid_char.best_params_,
    },
])

out_path = RUN_DIR / "gridsearch_summary.csv"
out.to_csv(out_path, index=False)
print("Saved:", out_path.relative_to(ROOT))
out

In [32]:
best_model = grid_word.best_estimator_  # כי word ניצח
best_model.fit(train_df_num, train_df_num["label"].values)

# evaluate on val/test using your existing helper
X_val, y_val   = val_df_num,  val_df_num["label"].values
X_test, y_test = test_df_num, test_df_num["label"].values

val_metrics  = evaluate_split("val",  best_model, X_val,  y_val)
test_metrics = evaluate_split("test", best_model, X_test, y_test)

print("\nVAL:", val_metrics)
print("\nTEST:", test_metrics)


val metrics:
  accuracy = 0.8406
  f1       = 0.8297
  precision= 0.8209
  recall   = 0.8386
  confusion matrix:
[[2907  544]
 [ 480 2494]]

test metrics:
  accuracy = 0.8373
  f1       = 0.8284
  precision= 0.8107
  recall   = 0.8468
  confusion matrix:
[[2861  590]
 [ 457 2527]]

VAL: SplitMetrics(split='val', accuracy=0.8406225680933852, f1=0.8296739853626082, precision=0.8209348255431205, recall=0.8386012104909213)

TEST: SplitMetrics(split='test', accuracy=0.8372960372960373, f1=0.8283887887231601, precision=0.8107154315046519, recall=0.8468498659517426)


In [ ]:
import pandas as pd

rows = [
    {"split": "val",  "accuracy": val_metrics.accuracy,  "f1": val_metrics.f1,
     "precision": val_metrics.precision, "recall": val_metrics.recall},
    {"split": "test", "accuracy": test_metrics.accuracy, "f1": test_metrics.f1,
     "precision": test_metrics.precision, "recall": test_metrics.recall},
]
df_out = pd.DataFrame(rows)

out_path = RUN_DIR / "bestmodel_val_test.csv"
df_out.to_csv(out_path, index=False)
print("Saved:", out_path.relative_to(ROOT))
df_out